In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
SCHEMA = REPO_ROOT / "database" / "create_schema.sql"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from cscaf.framework import ECSF
from cscaf.loading import FrameworkLoader
from cscaf.storage import Storage
from cscaf.study import Study
from cscaf.text import TextPipeline

study = Study.from_config()
storage = Storage.of(study, read_only=False)
if storage.database.create(SCHEMA):
    print("created:", storage.path.relative_to(REPO_ROOT))

In [ ]:
text = TextPipeline(study)
framework = ECSF.of(study)
print("ECSF file:", framework.path.relative_to(REPO_ROOT))
print("database :", storage.path.relative_to(REPO_ROOT))
print("model    :", text.model_name, "| stored as", text.embedding_dtype)

In [ ]:
TABLES = ["rf_section", "rf_subsection", "rf_sentence", "rf_subsection_sentence", "view_rf_sentences"]
{table: storage.count_rows(table) for table in TABLES}

In [ ]:
FrameworkLoader(study, storage, text).load(framework)

In [ ]:
counts = {table: storage.count_rows(table) for table in TABLES}
print(counts)
expected = study.section("reference_framework")
assert counts == {"rf_section": 12, "rf_subsection": 24, "rf_sentence": expected["expected_statements"],
                  "rf_subsection_sentence": expected["expected_rows"],
                  "view_rf_sentences": expected["expected_rows"]}, counts
ids, matrix, model = storage.framework.statement_embeddings(study.run_id)
print(f"{len(ids)} statements, embedding matrix {matrix.shape} {matrix.dtype} ({model}) | model dimension: "
      f"{text.embedding_dimension}")